# Machine Learning Case Study: BMI & Weight Category Classification
### Multiclass Classification of Body Mass Index (BMI) & Weight Status

This Jupyter Notebook demonstrates an end-to-end Machine Learning pipeline:
1. **Dataset Loading**: Ingesting BMI body measurement records.
2. **Exploratory Data Analysis**: Inspecting feature distributions and categories.
3. **Preprocessing & Splitting**: One-hot encoding for categorical variables with stratified train-test split.
4. **Model Training**: Scikit-Learn Pipeline with Random Forest Classifier.
5. **Model Evaluation**: Accuracy score, Precision, Recall, F1-Score, and Confusion Matrix.
6. **Model Artifact Export**: Saving the complete inference pipeline to `model.joblib`.
7. **Inference Demonstration**: Generating predictions on sample individual records.

In [1]:
# 1. Import dependencies
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

### 2. Load Dataset
Supports local execution or Google Colab execution with automatic fallback to the GitHub raw URL.

In [2]:
# 2. Load Dataset
dataset_path = 'bmi_dataset.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'https://raw.githubusercontent.com/SubraT933/ca-ai/main/bmi_dataset.csv'

df = pd.read_csv(dataset_path)
print(f'Dataset loaded successfully. Shape: {df.shape}')
df.head()

Dataset loaded successfully. Shape: (108, 7)


,ID,Age,Gender,Height,Weight,BMI,Label
0,1,25,Male,175,80,25.3,Normal Weight
1,2,30,Female,160,60,22.5,Normal Weight
2,3,35,Male,180,90,27.3,Overweight
3,4,40,Female,150,50,20.0,Underweight
4,5,45,Male,190,100,31.2,Obese


### 3. Exploratory Data Analysis

In [3]:
# 3. Target Distribution & Missing Values Check
print('Target Label Distribution:')
print(df['Label'].value_counts())
print('\nMissing Values Check:')
print(df.isnull().sum())
print('\nSummary Statistics:')
print(df[['Age', 'Height', 'Weight', 'BMI']].describe())

Target Label Distribution:
Label
Underweight      47
Normal Weight    29
Overweight       20
Obese            12
Name: count, dtype: int64

Missing Values Check:
ID        0
Age       0
Gender    0
Height    0
Weight    0
BMI       0
Label     0
dtype: int64

Summary Statistics:
              Age      Height      Weight         BMI
count  108.000000  108.000000  108.000000  108.000000
mean    46.555556  166.574074   59.490741   20.549074
std     24.720620   27.873615   28.856233    7.583818
min     11.000000  120.000000   10.000000    3.900000
25%     27.000000  140.000000   35.000000   16.700000
50%     42.500000  175.000000   55.000000   21.200000
75%     59.250000  190.000000   85.000000   26.100000
max    112.000000  210.000000  120.000000   37.200000


### 4. Feature Selection & Train-Test Split

In [4]:
# 4. Feature selection and train/test split
features = ['Age', 'Gender', 'Height', 'Weight', 'BMI']
X = df[features]
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Features: {features}')
print(f'Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}')

Features: ['Age', 'Gender', 'Height', 'Weight', 'BMI']
Training samples: 86, Testing samples: 22


### 5. Build Pipeline & Train Random Forest Model

In [5]:
# 5. Construct Preprocessing & Model Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Gender']),
        ('num', 'passthrough', ['Age', 'Height', 'Weight', 'BMI'])
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)
print('Pipeline constructed and trained successfully!')

Pipeline constructed and trained successfully!


### 6. Evaluate Model Performance

In [6]:
# 6. Evaluate on Test Set
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {acc * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 95.45%

Classification Report:
               precision    recall  f1-score   support

Normal Weight       0.86      1.00      0.92         6
        Obese       1.00      1.00      1.00         2
   Overweight       1.00      0.75      0.86         4
  Underweight       1.00      1.00      1.00        10

     accuracy                           0.95        22
    macro avg       0.96      0.94      0.95        22
 weighted avg       0.96      0.95      0.95        22

Confusion Matrix:
[[ 6  0  0  0]
 [ 0  2  0  0]
 [ 1  0  3  0]
 [ 0  0  0 10]]


### 7. Export Model Pipeline to model.joblib

In [7]:
# 7. Save model artifact
joblib.dump(model, 'model.joblib')
print('Model pipeline exported to model.joblib successfully!')

Model pipeline exported to model.joblib successfully!


### 8. Test Inference with Sample Profiles

In [8]:
# 8. Test loaded model pipeline
loaded_model = joblib.load('model.joblib')

sample_normal = pd.DataFrame([{'Age': 25, 'Gender': 'Male', 'Height': 175, 'Weight': 80, 'BMI': 25.3}])
sample_underweight = pd.DataFrame([{'Age': 15, 'Gender': 'Male', 'Height': 175, 'Weight': 45, 'BMI': 18.7}])
sample_obese = pd.DataFrame([{'Age': 45, 'Gender': 'Male', 'Height': 190, 'Weight': 100, 'BMI': 31.2}])

print(f'Sample 1 Prediction: {loaded_model.predict(sample_normal)[0]}')
print(f'Sample 2 Prediction: {loaded_model.predict(sample_underweight)[0]}')
print(f'Sample 3 Prediction: {loaded_model.predict(sample_obese)[0]}')

Sample 1 Prediction: Normal Weight
Sample 2 Prediction: Underweight
Sample 3 Prediction: Obese
